# KW — Lakehouse Schema

Creates Delta tables in the `KW_lh` Lakehouse with explicit schemas, partitioning, and column comments. Run this once against a clean Lakehouse before running the ingestion notebook.

**Bronze ↔ Silver convention**: this notebook builds the **silver** (curated) layer directly because the source CSVs are already well shaped. If you want a bronze landing zone, replicate the CREATE statements with `_raw` suffixes and `STRING` types.

In [ ]:
LAKEHOUSE = "KW_lh"
spark.sql(f"USE {LAKEHOUSE}")

## Reference / dimension tables

In [ ]:
ddl = [
    """CREATE TABLE IF NOT EXISTS dim_warehouse (
        warehouse_id STRING NOT NULL,
        name STRING,
        state STRING,
        address STRING,
        city STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        size_sqm INT,
        warehouse_type STRING,
        manager_employee_id STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_store (
        store_id STRING NOT NULL,
        name STRING,
        site_name STRING,
        city STRING,
        state STRING,
        address STRING,
        postcode STRING,
        latitude DOUBLE,
        longitude DOUBLE,
        size_sqm INT,
        opened_date DATE,
        manager_employee_id STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_fridge (
        fridge_id STRING NOT NULL,
        store_id STRING,
        code_name STRING,
        unit_type STRING,
        model STRING,
        min_safe_temp_c DOUBLE,
        max_safe_temp_c DOUBLE,
        install_date DATE,
        status STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_supplier (
        supplier_id STRING NOT NULL,
        name STRING,
        country STRING,
        contact_email STRING,
        lead_time_days INT,
        quality_score DOUBLE,
        tier STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_product (
        product_id STRING NOT NULL,
        sku STRING,
        name STRING,
        category STRING,
        storage_type STRING,
        unit_cost DOUBLE,
        unit_price DOUBLE,
        tax_rate DOUBLE,
        is_chilled BOOLEAN,
        is_frozen BOOLEAN,
        supplier_id STRING,
        weight_kg DOUBLE,
        buffer_stock INT
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_truck (
        truck_id STRING NOT NULL,
        rego STRING,
        model STRING,
        capacity_kg INT,
        tare_weight_kg INT,
        home_warehouse_id STRING,
        temperature_zone STRING,
        status STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_driver (
        driver_id STRING NOT NULL,
        employee_id STRING,
        name STRING,
        license_class STRING,
        license_issued DATE,
        home_warehouse_id STRING,
        shift STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_route (
        route_id STRING NOT NULL,
        origin_warehouse_id STRING,
        destination_store_id STRING,
        distance_km DOUBLE,
        estimated_duration_hours DOUBLE,
        route_type STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_forklift (
        forklift_id STRING NOT NULL,
        warehouse_id STRING,
        model STRING,
        capacity_kg INT,
        last_service_date DATE,
        status STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_employee (
        employee_id STRING NOT NULL,
        name STRING,
        role STRING,
        department STRING,
        location_id STRING,
        location_type STRING,
        employment_type STRING,
        hire_date DATE,
        base_salary_aud INT,
        manager_employee_id STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_customer (
        customer_id STRING NOT NULL,
        name STRING,
        email STRING,
        date_of_birth DATE,
        gender STRING,
        postcode STRING,
        state STRING,
        joined_date DATE,
        loyalty_tier STRING,
        preferred_channel STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_campaign (
        campaign_id STRING NOT NULL,
        name STRING,
        channel STRING,
        start_date DATE,
        end_date DATE,
        budget_aud INT,
        target_segment STRING,
        status STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS dim_promotion (
        promotion_id STRING NOT NULL,
        product_id STRING,
        promo_price DOUBLE,
        start_date DATE,
        end_date DATE,
        channel STRING
    ) USING DELTA""",
]
for stmt in ddl:
    spark.sql(stmt)
print(f'Created {len(ddl)} dimension tables.')

## Fact tables (no partitioning)

In [ ]:
fact_ddl = [
    """CREATE TABLE IF NOT EXISTS fact_store_stock (
        store_id STRING,
        product_id STRING,
        qty_on_hand INT,
        reorder_point INT,
        last_restocked_date DATE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_inwards_goods (
        receipt_id STRING,
        warehouse_id STRING,
        supplier_id STRING,
        product_id STRING,
        qty_in INT,
        unit_cost DOUBLE,
        receipt_date DATE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_shipments (
        shipment_id STRING,
        warehouse_id STRING,
        store_id STRING,
        ship_date DATE,
        total_weight_kg DOUBLE,
        service_level STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_shipment_lines (
        shipment_id STRING,
        product_id STRING,
        qty INT,
        line_weight_kg DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_pick_events (
        pick_id STRING,
        warehouse_id STRING,
        worker_employee_id STRING,
        pick_date DATE,
        items_picked INT,
        errors INT,
        duration_minutes DOUBLE,
        pick_rate_items_per_hour DOUBLE,
        is_priority_customer BOOLEAN,
        destination_store_id STRING,
        tare_weight_kg INT,
        bulk_replacement_minutes DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_trips (
        trip_id STRING,
        truck_id STRING,
        driver_id STRING,
        route_id STRING,
        departure_time TIMESTAMP,
        arrival_time TIMESTAMP,
        actual_duration_hours DOUBLE,
        status STRING,
        cargo_weight_kg DOUBLE,
        fuel_l_per_100km DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_sales (
        sale_id STRING,
        store_id STRING,
        sale_date DATE,
        sale_time TIMESTAMP,
        customer_id STRING,
        salesperson_employee_id STRING,
        subtotal DOUBLE,
        tax_total DOUBLE,
        total DOUBLE,
        payment_method STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_sale_lines (
        sale_id STRING,
        product_id STRING,
        qty INT,
        unit_price DOUBLE,
        tax_rate DOUBLE,
        line_total DOUBLE,
        line_tax DOUBLE,
        profit DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_employee_skills (
        skill_record_id STRING,
        employee_id STRING,
        skill_name STRING,
        certified_date DATE,
        expiry_date DATE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_performance_kpis (
        employee_id STRING,
        period_yyyymm STRING,
        kpi_score DOUBLE,
        absenteeism_days INT,
        turnover_flag BOOLEAN
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_training_records (
        training_id STRING,
        employee_id STRING,
        course_name STRING,
        completion_date DATE,
        priority STRING,
        status STRING
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_campaign_performance (
        campaign_id STRING,
        date DATE,
        impressions INT,
        clicks INT,
        conversions INT,
        revenue_aud DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_energy_consumption (
        store_id STRING,
        date DATE,
        kwh DOUBLE,
        cost_aud DOUBLE
    ) USING DELTA""",
    """CREATE TABLE IF NOT EXISTS fact_wastage (
        wastage_id STRING,
        store_id STRING,
        product_id STRING,
        date DATE,
        qty INT,
        cost_aud DOUBLE,
        reason STRING
    ) USING DELTA""",
]
for stmt in fact_ddl:
    spark.sql(stmt)
print(f'Created {len(fact_ddl)} fact tables.')

## Date dimension (for star schema joins)

In [ ]:
from pyspark.sql.functions import explode, sequence, to_date, lit, year, month, dayofmonth, dayofweek, date_format
df_dates = (spark.range(1).select(
    explode(sequence(to_date(lit('2024-01-01')), to_date(lit('2027-12-31')))).alias('date'))
    .withColumn('year', year('date'))
    .withColumn('month', month('date'))
    .withColumn('day', dayofmonth('date'))
    .withColumn('day_of_week', dayofweek('date'))
    .withColumn('month_name', date_format('date', 'MMMM'))
    .withColumn('is_weekend', dayofweek('date').isin([1,7]))
    .withColumn('financial_year', date_format('date', 'yyyy'))
)
df_dates.write.mode('overwrite').format('delta').saveAsTable('dim_date')
print('dim_date built.')

## Eventhouse / KQL tables (real-time)

These tables live in the **Eventhouse**, not the Lakehouse. Create them in your Eventhouse query editor (Kusto):

```kql
.create table fridge_telemetry (
    event_time: datetime,
    store_id: string,
    fridge_id: string,
    unit_type: string,
    temperature_c: real,
    humidity_pct: real,
    door_open: bool
)

.create table truck_telemetry (
    event_time: datetime,
    truck_id: string,
    trip_id: string,
    origin_warehouse_id: string,
    destination_store_id: string,
    latitude: real,
    longitude: real,
    speed_kmh: real,
    heading_deg: real,
    cargo_temp_c: real,
    progress_pct: real
)

// 30-day retention is sensible for IoT demos
.alter-merge table fridge_telemetry policy retention softdelete = 30d
.alter-merge table truck_telemetry  policy retention softdelete = 30d
```

Set up an **Eventstream** to map Event Hub → these KQL tables (see README).